# GLITCH ASE 2022: CPU-only reproduction
Use Colab free with a **CPU runtime**. The first cell downloads this standalone public repository, including the original source, Puppet parser egg, annotations, measured results and frozen correction. One historical oracle sample containing a private key is not published in Git: the evaluation cell restores it locally from the verified author image (1.82 GB).
The recorded benchmark was run locally on the Mac. Ruby/platform differences can affect parsing. Audit failure is a signal to investigate, not permission to silently claim an exact match. No IaC scripts are executed.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, subprocess, sys, urllib.request
destination = Path('/content').resolve()
archive = destination / 'glitch-cloud-security.zip'
urllib.request.urlretrieve('https://github.com/SimoneRodriguess/glitch-cloud-security/archive/refs/heads/main.zip', archive)
with zipfile.ZipFile(archive) as bundle:
    for item in bundle.infolist():
        if not (destination / item.filename).resolve().is_relative_to(destination):
            raise ValueError('Unsafe archive path')
    bundle.extractall(destination)
    project_folder = next((destination / name).parent.parent for name in bundle.namelist() if name.endswith('/glitchstudy/command_rule.py'))
%cd $project_folder


In [ ]:
import shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'venv', '.venv', '--python', '3.10'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', '.venv/bin/python', '-r', 'requirements-lock.txt'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', '.venv/bin/python', '--no-deps', '-e', 'artifact/GLITCH'], check=True)
if shutil.which('ruby') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ruby'], check=True)
subprocess.run(['ruby', '-rripper', '-e', 'puts \"Ruby and Ripper OK\"'], check=True)
subprocess.run(['.venv/bin/python', '-m', 'pytest', '-q'], check=True)


If Ruby is absent, install it in the Colab runtime with `apt-get update` and `apt-get install -y ruby`. Do not install/run Ansible, Chef or Puppet deployment engines. The Python 3.10 virtual environment is separate from the notebook kernel.

In [ ]:
def run_checked(*arguments):
    subprocess.run(['.venv/bin/python', *arguments], check=True)
run_checked('prepare_data.py')
run_checked('-u', '-m', 'glitchstudy.baseline')
run_checked('-u', '-m', 'glitchstudy.extension', 'development')
run_checked('-m', 'glitchstudy.extension', 'freeze')
run_checked('-u', '-m', 'glitchstudy.extension', 'test')
run_checked('-m', 'glitchstudy.audit')
run_checked('-m', 'glitchstudy.report')


In [ ]:
from IPython.display import Markdown, display, Image
display(Markdown(Path('results/REPORT.md').read_text()))
display(Image(filename='results/chef_holdout.png'))
import shutil
shutil.make_archive('/content/glitch-results', 'zip', root_dir='.', base_dir='results')
files.download('/content/glitch-results.zip')
shutil.make_archive('/content/glitch-paper-source', 'zip', root_dir='.', base_dir='paper')
files.download('/content/glitch-paper-source.zip')


## Manuscript
The supplied PDF records the audited Mac run. A Colab rerun updates the generated LaTeX tables but does not rebuild the PDF automatically. Compile `paper/main.tex` with its generated file, bibliography and figure on Overleaf using IEEEtran. Replace author details. Replaying the frozen holdout is reproducibility, not a fresh independent experiment.